# 🔮 KuroNeko TR v3 — Phi-4 Mini Turkish Fine-tune
**Platform:** Kaggle 2xT4 | **Model:** Phi-4-mini-instruct 3.8B | **Method:** QLoRA 4-bit  
**Data:** ~90K Turkish (70K HF + 20K synthetic) | **Epoch:** 3

In [ ]:
# CELL 1: KURULUM
!pip install -U transformers==4.49.0 trl==0.11.4 accelerate==0.34.2 peft==0.13.2 bitsandbytes==0.45.3 datasets==3.0.0 sentencepiece protobuf
import os, torch
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if torch.cuda.is_available(): print('GPU: ' + torch.cuda.get_device_name(0))
else: print('HATA: GPU yok')

In [ ]:
# CELL 1b: IMPORT CHECK
import transformers, bitsandbytes
print('transformers=' + transformers.__version__)
print('bitsandbytes=' + bitsandbytes.__version__)

In [ ]:
# CELL 2: HF TOKEN (Kaggle Secrets → env → empty)
import os
from huggingface_hub import login
HF_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN: Kaggle Secrets OK')
except Exception as e:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if HF_TOKEN:
        print('HF_TOKEN: environment variable OK')
    else:
        print('HF_TOKEN: bulunamadi, eger sonra girerseniz push yapilabilir')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    login(token=HF_TOKEN)
    print('HF login OK')
else:
    print('HF login atlandi (push yapilmayacak)')

In [ ]:
# CELL 3: GPU CHECK
import torch
print('CUDA: ' + str(torch.cuda.is_available()))
print('GPU sayisi: ' + str(torch.cuda.device_count()))
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print('  ' + str(i) + ': ' + torch.cuda.get_device_name(i) + ' (' + str(round(p.total_memory/1e9,1)) + 'GB)')

In [ ]:
# CELL 4: SYSTEM PROMPT + HF DATASET (~70K)
SYSTEM_PROMPT = 'Sen KuroNeko TR, Turkce konusan yapay zeka asistanisin. Sorulara detayli, adim adim, dogru cevaplar ver.'
from datasets import load_dataset
import random, os
random.seed(42)
os.makedirs('/kaggle/working/data', exist_ok=True)
all_data = []
def add(q, a):
    q, a = str(q).strip(), str(a).strip()
    if q and a and len(q) > 5 and len(a) > 5:
        all_data.append({'q': q[:800], 'a': a[:800]})
def load_ds(name, q, a, ctx=None, limit=15000):
    try:
        ds = load_dataset(name, split='train', cache_dir='/kaggle/working/hf_cache')
        if len(ds) > limit: ds = ds.shuffle(42).select(range(limit))
        cnt = 0
        for row in ds:
            qq = str(row.get(q, '')).strip()
            aa = str(row.get(a, '')).strip()
            if ctx and row.get(ctx):
                c = str(row[ctx]).strip()
                if c: qq = c + '\n\n' + qq
            add(qq, aa); cnt += 1
        print('  OK ' + name + ': ' + str(cnt))
    except Exception as e:
        print('  SKIP ' + name + ': ' + str(e)[:80])
print('Phase 1: HF...')
load_ds('merve/turkish_instructions', 'instruction', 'output', 'input', 15000)
load_ds('ucekmez/OpenOrca-tr', 'question', 'response', None, 15000)
load_ds('atasoglu/databricks-dolly-15k-tr', 'instruction', 'response', 'context', 15000)
load_ds('ytu-ce-cosmos/gsm8k_tr', 'question', 'answer', None, 8000)
load_ds('beratcmn/no_robots_turkish', 'instruction', 'output', None, 8000)
load_ds('beratcmn/lima-tr', 'instruction', 'output', None, 5000)
load_ds('nisancoskun/turkish_general_knowledge_qa', 'question', 'answer', None, 8000)
load_ds('umarigan/openhermes_tr', 'instruction', 'output', None, 5000)
print('HF toplam: ' + str(len(all_data)))

In [ ]:
# CELL 5: SYNTHETIC DATA (~20K, CoT format, Turkce karakterli)
import random as rnd
rnd.seed(42)
print('Phase 2: Sentetik...')

# === TURK TARIHI (495) ===
tt = [('Ataturk hangi yilda Samsun a cikti?\nA) 1918 B) 1919 C) 1920 D) 1921',
  'Adim adim:\nA) 1918 -> Yanlis, Mondros Muteskesi\nC) 1920 -> Yanlis, TBMM acildi\nD) 1921 -> Yanlis, Sakarya Savasi\nB) 1919 -> Dogru, 19 Mayis 1919\nCevap: B) 1919'),
 ('Osmanli ne zaman kuruldu?\nA) 1299 B) 1453 C) 1517 D) 1922',
  'Adim adim:\nB) 1453 -> Istanbul fethi\nC) 1517 -> Halifelik\nD) 1922 -> Sona erdi\nA) 1299 -> Dogru, Osman Bey\nCevap: A) 1299'),
 ('I.Dunya Savasi ne zaman basladi?\nA) 1914 B) 1918 C) 1939 D) 1945',
  'Adim adim:\nB) 1918 -> Savas sonu\nC) 1939 -> II.DS baslangici\nD) 1945 -> II.DS sonu\nA) 1914 -> Dogru, 28 Temmuz 1914\nCevap: A) 1914'),
 ('Kurtulus Savasi ne zaman basladi?\nA) 1918 B) 1919 C) 1920 D) 1923',
  'Adim adim:\nA) 1918 -> Mondros\nC) 1920 -> TBMM\nD) 1923 -> Cumhuriyet\nB) 1919 -> Dogru, 19 Mayis 1919\nCevap: B) 1919'),
 ('Cumhuriyet ne zaman ilan edildi?\nA) 1919 B) 1920 C) 1922 D) 1923',
  'Adim adim:\nA) 1919 -> Kurtulus basladi\nB) 1920 -> TBMM acildi\nC) 1922 -> Saltanat kaldirildi\nD) 1923 -> Dogru, 29 Ekim 1923\nCevap: D) 1923'),
 ('Canakkale Savasi ne zaman?\nA) 1914-1915 B) 1915-1916 C) 1914-1918 D) 1919-1923',
  'Adim adim:\nB) 1915-1916 -> Yanlis\nC) 1914-1918 -> I.DS suresi\nD) 1919-1923 -> Kurtulus Savasi\nA) 1914-1915 -> Dogru\nCevap: A) 1914-1915'),
 ('Ataturk hangi yilda vefat etti?\nA) 1936 B) 1937 C) 1938 D) 1939',
  'Adim adim:\nA) 1936 -> Yanlis\nB) 1937 -> Yanlis\nD) 1939 -> II.DS basladi\nC) 1938 -> Dogru, 10 Kasim 1938\nCevap: C) 1938'),
 ('Istanbul ne zaman fethedildi?\nA) 1299 B) 1453 C) 1517 D) 1922',
  'Adim adim:\nA) 1299 -> Osmanli kurulusu\nC) 1517 -> Halifelik\nD) 1922 -> Saltanat kaldirildi\nB) 1453 -> Dogru, Fatih Sultan Mehmet\nCevap: B) 1453'),
 ('Tanzimat Fermani ne zaman?\nA) 1839 B) 1876 C) 1908 D) 1923',
  'Adim adim:\nB) 1876 -> I.Donem\nC) 1908 -> II.Donem\nD) 1923 -> Cumhuriyet\nA) 1839 -> Dogru, 3 Kasim 1839\nCevap: A) 1839'),
 ('Balkan Savaslari ne zaman?\nA) 1911-1912 B) 1912-1913 C) 1913-1914 D) 1914-1918',
  'Adim adim:\nA) 1911-1912 -> Italyan-Turk Savasi\nC) 1913-1914 -> Yanlis\nD) 1914-1918 -> I.DS\nB) 1912-1913 -> Dogru\nCevap: B) 1912-1913'),
 ('Misak-i Milli ne zaman?\nA) 1919 B) 1920 C) 1921 D) 1922',
  'Adim adim:\nA) 1919 -> Son Osmali Meclis\nC) 1921 -> Saltanat kaldirildi\nD) 1922 -> Cumhuriyet oncesi\nB) 1920 -> Dogru, 28 Ocak 1920\nCevap: B) 1920'),
 ('Saltanat ne zaman kaldirildi?\nA) 1919 B) 1920 C) 1922 D) 1923',
  'Adim adim:\nA) 1919 -> Kurtulus basladi\nB) 1920 -> Misak-i Milli\nD) 1923 -> Cumhuriyet ilan\nC) 1922 -> Dogru, 1 Kasim 1922\nCevap: C) 1922'),
 ('Harf Devrimi ne zaman?\nA) 1923 B) 1926 C) 1928 D) 1930',
  'Adim adim:\nA) 1923 -> Cumhuriyet\nB) 1926 -> Yanlis\nD) 1930 -> Yanlis\nC) 1928 -> Dogru, 1 Kasim 1928\nCevap: C) 1928'),
 ('Sakarya Meydan Muharebesi ne zaman?\nA) 1919 B) 1920 C) 1921 D) 1922',
  'Adim adim:\nA) 1919 -> Kurtulus baslangici\nB) 1920 -> TBMM acildi\nD) 1922 -> Izmir kurtarildi\nC) 1921 -> Dogru, Agustos-Eylul 1921\nCevap: C) 1921'),
 ('Lozan Antlasmasi ne zaman?\nA) 1920 B) 1921 C) 1922 D) 1923',
  'Adim adim:\nA) 1920 -> Misak-i Milli\nB) 1921 -> Sakarya Savasi\nC) 1922 -> Saltanat kaldirildi\nD) 1923 -> Dogru, 24 Temmuz 1923\nCevap: D) 1923'),
 ('TBMM ne zaman acildi?\nA) 1919 B) 1920 C) 1921 D) 1922',
  'Adim adim:\nA) 1919 -> Son Osmali Meclis\nC) 1921 -> Ankara\nD) 1922 -> Saltanat kaldirildi\nB) 1920 -> Dogru, 23 Nisan 1920\nCevap: B) 1920')]
for q,a in tt * 33: add(q,a)
print('  Turk Tarihi: ' + str(len(tt)*33))

# === DUNYA TARIHI (275) ===
dt = [('Fransiz Devrimi ne zaman?\nA) 1776 B) 1789 C) 1804 D) 1815',
  'Adim adim:\nA) 1776 -> ABD Bagimsizlik\nC) 1804 -> Napolyon\nD) 1815 -> Napolyon yenildi\nB) 1789 -> Dogru, 14 Temmuz 1789\nCevap: B) 1789'),
 ('Amerikan Bagimsizlik ne zaman?\nA) 1776 B) 1789 C) 1804 D) 1812',
  'Adim adim:\nB) 1789 -> Fransiz Devrimi\nC) 1804 -> Napolyon\nD) 1812 -> 1812 Savasi\nA) 1776 -> Dogru, 4 Temmuz 1776\nCevap: A) 1776'),
 ('Berlin Duvari ne zaman yikildi?\nA) 1987 B) 1988 C) 1989 D) 1990',
  'Adim adim:\nA) 1987 -> Reagan konusmasi\nB) 1988 -> Yanlis\nD) 1990 -> Almanya birlesti\nC) 1989 -> Dogru, 9 Kasim 1989\nCevap: C) 1989'),
 ('Kuban Fuzesi Krizi ne zaman?\nA) 1959 B) 1960 C) 1961 D) 1962',
  'Adim adim:\nB) 1960 -> Yanlis\nC) 1961 -> Berlin Duvari\nD) 1962 -> Yanlis\nA) 1959 -> Yanlis\nCevap: D) 1962'),
 ('I.DS ne zaman bitti?\nA) 1914 B) 1916 C) 1918 D) 1919',
  'Adim adim:\nA) 1914 -> Basladi\nB) 1916 -> Verdun\nD) 1919 -> Versay\nC) 1918 -> Dogru, 11 Kasim 1918\nCevap: C) 1918'),
 ('II.DS ne zaman basladi?\nA) 1914 B) 1936 C) 1939 D) 1941',
  'Adim adim:\nA) 1914 -> I.DS\nB) 1936 -> Ispanya Ic Savasi\nD) 1941 -> Pearl Harbor\nC) 1939 -> Dogru, 1 Eylul 1939\nCevap: C) 1939'),
 ('II.DS ne zaman bitti?\nA) 1943 B) 1944 C) 1945 D) 1946',
  'Adim adim:\nA) 1943 -> Italya teslim\nB) 1944 -> D-Day\nD) 1946 -> Savas sonrasi\nC) 1945 -> Dogru, 2 Eylul 1945\nCevap: C) 1945'),
 ('Roma Imparatorlugu ne zaman yikildi?\nA) 476 B) 1453 C) 1492 D) 1517',
  'Adim adim:\nB) 1453 -> Dogu Roma\nC) 1492 -> Amerika\nD) 1517 -> Reform\nA) 476 -> Dogru, Bati Roma\nCevap: A) 476'),
 ('Sanayi Devrimi nerede basladi?\nA) Fransa B) Almanya C) Ingiltere D) ABD',
  'Adim adim:\nA) Fransa -> Fransiz Devrimi\nB) Almanya -> II.Sanayi\nD) ABD -> 20.yy\nC) Ingiltere -> Dogru, 18.yy\nCevap: C) Ingiltere'),
 ('Berlin ne zaman fethedildi?\nA) 1943 B) 1944 C) 1945 D) 1946',
  'Adim adim:\nA) 1943 -> Stalingrad\nB) 1944 -> D-Day\nD) 1946 -> Savas sonrasi\nC) 1945 -> Dogru, Mayis 1945\nCevap: C) 1945')]
for q,a in dt * 25: add(q,a)
print('  Dunya Tarihi: ' + str(len(dt)*25))

# === TURK COGRAFYASI (275) ===
cg = [('En buyuk gol?\nA) Egirdir B) Van C) Iznik D) Tuz',
  'Adim adim:\nA) Egirdir -> Ikinci\nC) Iznik -> Kucuk\nD) Tuz -> Tuz golu\nB) Van -> Dogru, en buyuk tatli su\nCevap: B) Van Golu'),
 ('En yuksek dag?\nA) Uludag B) Agri C) Erciyes D) Nemrut',
  'Adim adim:\nA) Uludag -> 2543m\nC) Erciyes -> 3916m\nD) Nemrut -> 2150m\nB) Agri -> Dogru, 5137m\nCevap: B) Agri Dagi'),
 ('En uzun nehir?\nA) Firat B) Dicle C) Kizilirmak D) Sakarya',
  'Adim adim:\nA) Firat -> 1350km sinir\nB) Dicle -> 1200km\nD) Sakarya -> 824km\nC) Kizilirmak -> Dogru, 1182km\nCevap: C) Kizilirmak'),
 ('Hangi sehir Akdeniz?\nA) Ankara B) Izmir C) Antalya D) Trabzon',
  'Adim adim:\nA) Ankara -> Ic Anadolu\nB) Izmir -> Ege\nD) Trabzon -> Karadeniz\nC) Antalya -> Dogru\nCevap: C) Antalya'),
 ('Kuzey siniri hangi deniz?\nA) Akdeniz B) Ege C) Karadeniz D) Marmara',
  'Adim adim:\nA) Akdeniz -> Guney\nB) Ege -> Bati\nD) Marmara -> Ic\nC) Karadeniz -> Dogru\nCevap: C) Karadeniz'),
 ('En dogu il?\nA) Kars B) Igdir C) Van D) Hakkari',
  'Adim adim:\nA) Kars -> Dogu ama en degil\nC) Van -> Dogu\nD) Hakkari -> Dogu\nB) Igdir -> Dogru, en dogu\nCevap: B) Igdir'),
 ('Bogazlar hangi denizleri birlestirir?\nA) Akdeniz-Ege B) Karadeniz-Marmara C) Ege-Marmara D) Akdeniz-Karadeniz',
  'Adim adim:\nA) Akdeniz-Ege -> Yanlis\nC) Ege-Marmara -> Canakkale\nD) Akdeniz-Karadeniz -> Yanlis\nB) Karadeniz-Marmara -> Dogru, Istanbul Bogazi\nCevap: B) Karadeniz-Marmara'),
 ('Ikinci buyuk gol?\nA) Van B) Egirdir C) Iznik D) Tuz',
  'Adim adim:\nA) Van -> En buyuk\nC) Iznik -> Kucuk\nD) Tuz -> Tuz golu\nB) Egirdir -> Dogru, ikinci buyuk tatli su\nCevap: B) Egirdir Golu'),
 ('En kucuk il?\nA) Yalova B) Kilis C) Tunceli D) Bayburt',
  'Adim adim:\nB) Kilis -> 1676 km2\nC) Tunceli -> 7572 km2\nD) Bayburt -> 3652 km2\nA) Yalova -> Dogru, 847 km2\nCevap: A) Yalova'),
 ('Ikinci yuksek dag?\nA) Agri B) Uludag C) Erciyes D) Suphan',
  'Adim adim:\nA) Agri -> En buyuk\nB) Uludag -> 2543m\nC) Erciyes -> 3916m\nD) Suphan -> Dogru, 4058m\nCevap: D) Suphan Dagi')]
for q,a in cg * 25: add(q,a)
print('  Cografya: ' + str(len(cg)*25))

# === BIYOLOJI (275) ===
bio = [('Fotosentez?', 'Adim adim:\n1. Tanim: Bitkiler isigi enerjeye cevirir\n2. Kloroplast, klorofil\n3. 6CO2+6H2O+Isik->C6H12O6+6O2\n4. Fotosistem II/I\n5. Calvin dongu\n6. Oksijen, besin zinciri'),
 ('DNA?', 'Adim adim:\n1. Cift sarmal, nukleotidler\n2. A-T, G-C\n3. Genetik bilgi\n4. Replikasyon\n5. DNA->mRNA->Protein'),
 ('Hucresel solunum?', 'Adim adim:\n1. Glikoz->enerji\n2. Sitoplazma+Mitokondri\n3. C6H12O6+6O2->6CO2+6H2O+ATP\n4. Glikoliz, Krebs, Elektron tasimasi\n5. 36-38 ATP'),
 ('Kalitim?', 'Adim adim:\n1. DNA replikasyonu\n2. Meiosis: Gametler\n3. Fertilizasyon: Zigot\n4. Gen ifadesi\n5. Mendel yasari'),
 ('Evolusyon?', 'Adim adim:\n1. Turlerin degismesi\n2. Dogal secilim\n3. Mutasyon, rekombinasyon\n4. Uyum\n5. Fosil kanitlari'),
 ('Bagisiklik sistemi?', 'Adim adim:\n1. Vucudu savunma\n2. Dogal: Makrofaj, NK\n3. Edinsel: B ve T lenfosit\n4. Antikor\n5. Asi'),
 ('Sinir sistemi?', 'Adim adim:\n1. Noron: Temel birim\n2. Sinaps: Iletisim\n3. Nurotransmitter\n4. Beyin+Omurilik\n5. Refleks'),
 ('Kan dongusu?', 'Adim adim:\n1. Kalp: Pompa\n2. Atrium+Ventrikuler\n3. Sistemik+Pulmoner\n4. Arter+Ven+Kapiller\n5. Oksijen tasimasi'),
 ('Protein sentezi?', 'Adim adim:\n1. Transkripsiyon: DNA->mRNA\n2. Translasyon: mRNA->Protein\n3. tRNA: Aminoasit\n4. Ribozom\n5. Golgi: Paketleme'),
 ('Hucre bolunme?', 'Adim adim:\n1. Mitoz: 2 ozdes\n2. Mayoz: 4 yari\n3. Amitoz: Bakteri\n4. Kanser: Kontrolsuz mitoz')]
for q,a in bio * 25: add(q,a)
print('  Biyoloji: ' + str(len(bio)*25))

# === FIZIK (275) ===
fzk = [('Newton 1.yasa?', 'Adim adim:\n1. Eylemsizlik\n2. F=0 ise v=sabit\n3. Otobus ornegi\n4. Kutle hareket etmez'),
 ('Newton 2.yasa?', 'Adim adim:\n1. F=m x a\n2. F:Newton, m:kg, a:m/s2\n3. Araba ivmelenmesi'),
 ('Newton 3.yasa?', 'Adim adim:\n1. Etki-tepki\n2. F12=-F21\n3. Duvara itme\n4. Roket firlatma'),
 ('Enerji korunumu?', 'Adim adim:\n1. Enerji donusur\n2. Ep=mgh\n3. Ek=0.5mv2\n4. Sarkit oyunu'),
 ('Yercekimi ivmesi?', 'Adim adim:\n1. g=9.81 m/s2\n2. Dunya yuzeyinde\n3. Yukseltive azalir\n4. F=mg'),
 ('Isi?', 'Adim adim:\n1. Enerji transferi\n2. Joule veya Kalori\n3. Sicaklik\n4. Iletim, Tasinma, Isinma'),
 ('Ses?', 'Adim adim:\n1. Mekanik dalga\n2. Ortam gerekli\n3. Havada ~343 m/s\n4. 20-20000 Hz'),
 ('Elektrik akimi?', 'Adim adim:\n1. Elektron akisi\n2. Amper\n3. V=I x R\n4. P=V x I'),
 ('Isik?', 'Adim adim:\n1. Elektromanyetik dalga\n2. 300.000 km/s\n3. Dalga+Parcacik\n4. 400-700 nm'),
 ('E=mc2?', 'Adim adim:\n1. E=mc2\n2. Kucuk kutle=buyuk enerji\n3. Gunes\n4. Nukleer enerji')]
for q,a in fzk * 25: add(q,a)
print('  Fizik: ' + str(len(fzk)*25))

# === KIMYA (198) ===
kim = [('Su molekulu?', 'Adim adim:\n1. H2O\n2. 2H+1O\n3. Kovalent bag\n4. 100C kaynar\n5. 0C donar'),
 ('Asit?', 'Adim adim:\n1. H+ veren\n2. pH 0-7\n3. HCl, H2SO4\n4. Metal eritir\n5. Turnusol kirmizi'),
 ('Baz?', 'Adim adim:\n1. OH- veren\n2. pH 7-14\n3. NaOH, KOH\n4. Kaygan\n5. Turnusol mavi'),
 ('Periyodik tablo?', 'Adim adim:\n1. Elementler tablosu\n2. Atom numarasi\n3. 7 periyot, 18 grup\n4. H(1), He(2), Li(3)'),
 ('Kovalent bag?', 'Adim adim:\n1. Elektron paylasimi\n2. H2O, CO2\n3. Metal olmayan+Metal olmayan\n4. Kotu iletken'),
 ('Iyonik bag?', 'Adim adim:\n1. Elektron transferi\n2. NaCl, CaO\n3. Metal+Metal olmayan\n4. Suda iyi cozunur'),
 ('Yanma?', 'Adim adim:\n1. Oksijenle tepkime\n2. Yakit+O2->CO2+H2O\n3. Ekzotermik\n4. Motorlar'),
 ('Mol?', 'Adim adim:\n1. 6.022x10^23 parcacik\n2. n=m/M\n3. 1 mol H2O=18g')]
for q,a in kim * 22: add(q,a)
print('  Kimya: ' + str(len(kim)*22))

# === TURK EDEBIYATI (225) ===
edeb = [('Istanbul mehter marsi kime ait?', 'Adim adim:\n1. Umit Yasar Oguzcan\n2. Istanbul senin buyuk adin\n3. Turku\n4. Kultur mirasi\n5. Cevap: Umit Yasar Oguzcan'),
 ('Necip Fazil hangi eser?', 'Adim adim:\n1. Buyuk Dozu\n2. Hicbir Fikri yok\n3. Horoz\n4. Bir adam geldi\n5. Cevap: Buyuk Dozu'),
 ('Orhan Pamuk hangi eserle Nobel?', 'Adim adim:\n1. 2006 Nobel\n2. Benim Adim Kirmizi\n3. Turkiye\n4. Cevap: Benim Adim Kirmizi'),
 ('Yunus Emre hangi yy?', 'Adim adim:\n1. 1240-1321\n2. Selcuklu Donemi\n3. Risaletu n-Nushiyye\n4. Tasavvuf\n5. Cevap: 13.yy'),
 ('Cemal Sureya hangi akim?', 'Adim adim:\n1. Ikinci Yeni\n2. Soyut, sembolik\n3. Uzun Siradaki Oduncu Adam\n4. Modern Turk siiri\n5. Cevap: Ikinci Yeni'),
 ('Attila Ilhan hangi eser?', 'Adim adim:\n1. Siyah Pantolonlu Kadin\n2. Turk romancisi\n3. Modern hayat\n4. 1950\n5. Cevap: Siyah Pantolonlu Kadin'),
 ('Sabahattin Ali hangi eser?', 'Adim adim:\n1. Kuyucakli Yusuf\n2. Turk romancisi\n3. Koy hayati\n4. 1937\n5. Cevap: Kuyucakli Yusuf'),
 ('Omer Seyfettin hangi akim?', 'Adim adim:\n1. Milli Edebiyat\n2. Turkce, milli\n3. Gulyabani\n4. Hikaye\n5. Cevap: Milli Edebiyat'),
 ('Nazim Hikmet hangi eser?', 'Adim adim:\n1. Memleketimden Insan Manzaralari\n2. Epik siir\n3. 1930\n4. Cevap: Memleketimden Insan Manzaralari')]
for q,a in edeb * 25: add(q,a)
print('  Edebiyat: ' + str(len(edeb)*25))

# === FILOSOFI (198) ===
fel = [('Ozgurluk?', 'Adim adim:\n1. Kendi irade\n2. Negatif/Pozitif\n3. Sinirlar: haklar hukuk\n4. Konusma ozgurlugu\n5. Demokrasi temeli'),
 ('Adalet?', 'Adim adim:\n1. Hak ettigini verme\n2. Dagitim/Prosedurel/Duzeltici\n3. Rawls, Nozick\n4. Toplum duzeni'),
 ('Bilgi felsefesi?', 'Adim adim:\n1. JTB: Gerekceli Dogru Inanc\n2. Deneyim, akil\n3. Gettier problem\n4. Skeptisizm'),
 ('Ahlak?', 'Adim adim:\n1. Sonucculuk\n2. Deontoloji\n3. Erdem\n4. Bakim'),
 ('Varolusculuk?', 'Adim adim:\n1. Once varlik\n2. Anlami biz yaratiriz\n3. Sartre, Camus\n4. Bireysel sorumluluk'),
 ('Stoacilik?', 'Adim adim:\n1. Kabullenme\n2. Epiktet, Marcus Aurelius\n3. Ic huzur\n4. Stres yonetimi'),
 ('Epikuculuk?', 'Adim adim:\n1. Haz felsefesi\n2. Basit yasam\n3. Arkadaslik\n4. Dengeli yasam'),
 ('Rasyonalizm?', 'Adim adim:\n1. Akil temelli\n2. Descartes: Dusunuyorum oyleyse varim\n3. Bilim yontemi')]
for q,a in fel * 22: add(q,a)
print('  Felsefe: ' + str(len(fel)*22))

# === EKONOMI (198) ===
eko = [('Enflasyon?', 'Adim adim:\n1. Fiyat artisi\n2. Para arzi, talep\n3. TUFE, UFE\n4. Alinma gucu dususu\n5. Faiz artisi'),
 ('GSYIH?', 'Adim adim:\n1. Gayri Safi Yurtici Hasila\n2. C+I+G+(X-M)\n3. Nominal vs Reel\n4. Kisi basi GSYIH'),
 ('Arz ve talep?', 'Adim adim:\n1. Arz: Uretici\n2. Talep: Tuketici\n3. Denge\n4. Fiyat artar->talep azalir'),
 ('Faiz?', 'Adim adim:\n1. Paranin karsiligi\n2. Basit: P x r x t\n3. Bilesik: P x (1+r)^t\n4. Reel faiz=Nominal-Enflasyon'),
 ('Borsa?', 'Adim adim:\n1. Menkul kiymet alis-satis\n2. Hisse senedi\n3. BIST 100\n4. Bull/Bear market'),
 ('Doviz kuru?', 'Adim adim:\n1. TL/USD\n2. Arz-talep\n3. Merkez bankasi\n4. Enflasyon etkisi'),
 ('Kripto?', 'Adim adim:\n1. Dijital para\n2. Blockchain\n3. Bitcoin\n4. Yuksek risk'),
 ('Merkez bankasi?', 'Adim adim:\n1. Para politikasi\n2. TCMB, Fed, ECB\n3. Enflasyon kontrolu\n4. Faiz, rezerv')]
for q,a in eko * 22: add(q,a)
print('  Ekonomi: ' + str(len(eko)*22))

# === MATEMATIK CoT (1500) ===
for _ in range(250):
    a,b,c,d = rnd.randint(3,50), rnd.randint(3,50), rnd.randint(1,10), rnd.randint(1,10)
    add(str(a)+' TL/kg elma, '+str(b)+' TL/kg armut. '+str(c)+'kg+'+str(d)+'kg?',
        'Adim 1: Elma='+str(a*c)+' TL\nAdim 2: Armut='+str(b*d)+' TL\nAdim 3: Toplam='+str(a*c+b*d)+' TL\nCevap: '+str(a*c+b*d)+' TL')
for _ in range(250):
    u,k = rnd.randint(5,50), rnd.randint(3,40)
    add('Dikdortgen u='+str(u)+'cm k='+str(k)+'cm?', 'Adim 1: A='+str(u*k)+' cm2\nAdim 2: C='+str(2*(u+k))+' cm\nCevap: A='+str(u*k)+' C='+str(2*(u+k)))
for _ in range(250):
    t = rnd.randint(50,500); p = rnd.choice([5,10,15,20,25,30,40,50]); r = int(t*p/100)
    add(str(t)+' nin %'+str(p)+' si?', 'Adim 1: '+str(t)+'x'+str(p)+'/100\nAdim 2: ='+str(r)+'\nCevap: '+str(r))
for _ in range(250):
    v,h = rnd.randint(40,120), rnd.randint(1,12)
    add(str(v)+' km/h x '+str(h)+' saat?', 'Adim 1: Mesafe=HizxZaman\nAdim 2: '+str(v)+'x'+str(h)+'\nCevap: '+str(v*h)+' km')
for _ in range(250):
    nums = [rnd.randint(1,100) for _ in range(rnd.randint(3,7))]
    add(str(nums)+' ortalamasi?', 'Adim 1: Top='+str(sum(nums))+' n='+str(len(nums))+'\nAdim 2: O='+str(round(sum(nums)/len(nums),2))+'\nCevap: '+str(round(sum(nums)/len(nums),2)))
for _ in range(250):
    s,d = rnd.randint(1,20), rnd.randint(2,7)
    seq = [s+d*i for i in range(8)]
    add('Dizi '+str(seq[:6])+' ?', 'Adim 1: Fark='+str(d)+'\nAdim 2: Sonraki='+str(seq[6])+'\nCevap: '+str(seq[6]))
print('  Matematik CoT: 1500')

# === COK ADIMLI PROBLEM (800) ===
for _ in range(200):
    a,b,c = rnd.randint(10,100), rnd.randint(5,50), rnd.randint(2,10)
    add(str(a)+' gun 1 isci. '+str(b)+' isci kac gun? '+str(c)+' isci kac gun?',
        'Adim 1: Toplam is='+str(a)+' isci-gun\nAdim 2: '+str(b)+' isci='+str(round(a/b,1))+' gun\nAdim 3: '+str(c)+' isci='+str(round(a/c,1))+' gun\nCevap: '+str(b)+' isci='+str(round(a/b,1))+' g, '+str(c)+' isci='+str(round(a/c,1))+' g')
for _ in range(200):
    v1,v2,dist = rnd.randint(40,100), rnd.randint(40,100), rnd.randint(100,500)
    add(str(dist)+' km arayla birbirine dogru. '+str(v1)+' km/h ve '+str(v2)+' km/h. Kac saat karsilasir?',
        'Adim 1: Toplam hiz='+str(v1+v2)+' km/h\nAdim 2: Sure='+str(dist)+'/'+str(v1+v2)+'\nCevap: '+str(round(dist/(v1+v2),2))+' saat')
for _ in range(200):
    y,f = rnd.randint(5,50), rnd.randint(100,1000)
    add(str(f)+' TL. Once %'+str(y)+' zam, sonra %'+str(y)+' indirim. Son fiyat?',
        'Adim 1: Zamli='+str(round(f*(1+y/100),2))+'\nAdim 2: Indirimli='+str(round(f*(1+y/100)*(1-y/100),2))+'\nCevap: '+str(round(f*(1+y/100)*(1-y/100),2))+' TL')
for _ in range(200):
    a,b = rnd.randint(2,20), rnd.randint(2,20)
    add('Havuz '+str(a)+' saat A, '+str(b)+' saat B. Ikisi birlikte kac saat?',
        'Adim 1: A hizi=1/'+str(a)+'\nAdim 2: B hizi=1/'+str(b)+'\nAdim 3: Toplam='+str(round(1/a+1/b,4))+'\nCevap: '+str(round(1/(1/a+1/b),2))+' saat')
print('  Cok Adimli: 800')

# === KOD CoT (600) ===
kod = [('Python liste ters cevir?', 'Adim adim:\n1. reverse() in-place:\nliste=[1,2,3,4]\nliste.reverse()  #[4,3,2,1]\n2. Slicing:\nters=liste[::-1]\n3. reversed():\nters=list(reversed(liste))'),
 ('Python dict?', 'Adim adim:\n1. Literal: o={"ad":"Ali","yas":20}\n2. dict(): o=dict(ad="Ali",yas=20)\n3. Erisim: o["ad"], o.get("tel","yok")\n4. Guncelleme: o.update({"sehir":"Istanbul"})'),
 ('Python class?', 'Adim adim:\n1. class Hayvan:\n    def __init__(self,ad,yas):\n        self.ad=ad\n        self.yas=yas\n2. class Kopek(Hayvan):\n    def ses_cikar(self): return "Hav hav!"\n3. k=Kopek("Max",3)'),
 ('Git branch?', 'Adim adim:\n1. git checkout -b yeni\n2. git add .\n3. git commit -m "mesaj"\n4. git checkout main\n5. git merge yeni\n6. git branch -d yeni'),
 ('Docker yonetimi?', 'Adim adim:\n1. docker run -d --name web nginx\n2. -p 8080:80\n3. docker exec -it web bash\n4. docker logs web\n5. docker stop web\n6. docker rm web'),
 ('SQL injection?', 'Adim adim:\n1. cursor.execute(f"...{uid}") TEHLIKELI\n2. cursor.execute("...%s",(uid,)) GUCLU\n3. ORM: User.objects.filter(id=uid) EN IYI'),
 ('Python HTTP?', 'Adim adim:\nimport requests\n1. r=requests.get(url)\n2. r=requests.post(url,json=data)\n3. r.raise_for_status()\n4. headers={"Authorization":"Bearer TOKEN"}'),
 ('Linux izinler?', 'Adim adim:\n1. chmod 755 f.sh (rwxr-xr-x)\n2. chmod +x f.sh\n3. 4=Oku 2=Yaz 1=Calistir\n4. chown user:grup f'),
 ('JS async/await?', 'Adim adim:\n1. async function veriGetir(){\n    try{\n        const r=await fetch(url);\n        const d=await r.json();\n    }catch(e){\n        console.error(e);\n    }\n}\n2. await: Promise bekler\n3. try-catch: Hata yonetimi'),
 ('SQL JOIN?', 'Adim adim:\n1. INNER: eslesen\n2. LEFT: sol tum\n3. RIGHT: sag tum\n4. FULL: tum kayitlar\n5. CROSS: kombinasyon'),
 ('List comprehension?', 'Adim adim:\n1. [x**2 for x in range(10)]\n2. [x for x in l if x>5]\n3. {k:v**2 for k,v in s.items()}\n4. [[i*j for j in range(5)] for i in range(5)]'),
 ('Docker Compose?', 'Adim adim:\nversion: "3"\nservices:\n  web:\n    image: nginx\n    ports: ["8080:80"]\n  db:\n    image: postgres\ndocker-compose up -d')]
for q,a in kod * 50: add(q,a)
print('  Kod CoT: ' + str(len(kod)*50))

# === TURKCE KULTUR (400) ===
kultur = [('Turk mutfagi?', 'Adim adim:\n1. Kebap: Adana Urfa Iskender\n2. Corba: Mercimek tarhana\n3. Tatli: Baklava kunefe lokum\n4. Icecek: Turk kahvesi cay\n5. Kahvalti: Simit peynir zeytin'),
 ('UNESCO Miraslari?', 'Adim adim:\n1. Kapadokya\n2. Pamukkale\n3. Efes\n4. Nemrut Dagi\n5. Safranbolu\n6. Istanbul\n7. Gobeklitepe\n8. Catalhoyuk\n9. Bursa\n10. Diyarbakir'),
 ('Turk halk muzigi sanatcilari?', 'Adim adim:\n1. Aseik Veysel\n2. Neset Ertas\n3. Zeki Muren\n4. Baris Manco\n5. Sezen Aksu'),
 ('Turk edebiyati akimlari?', 'Adim adim:\n1. Tanzimat: Namik Kemal\n2. Servet-i Funun: Tevfik Fikret\n3. Milli Edebiyat: Omer Seyfettin\n4. Cumhuriyet: Nazim Hikmet Orhan Pamuk\n5. Ikinci Yeni: Cemal Sureya'),
 ('Festivalleri?', 'Adim adim:\n1. Istanbul Film\n2. Antalya Altin Portakal\n3. Konya Mevlana\n4. Edirne Kirkpinar\n5. Istanbul Caz'),
 ('Turk kahvesi yapimi?', 'Adim adim:\n1. Cezve kullanilir\n2. 1 fincan su + 1 kasik kahve\n3. Yavas pisirilir\n4. Kopuk olustugu karistirilir\n5. Fincana yavasce doldurulur'),
 ('Ebru sanati?', 'Adim adim:\n1. Kagit boyama sanati\n2. Su, kitre, boya\n3. Battal, gelenekli, hata, tarak\n4. UNESCO Kultur Mirasi'),
 ('Kilim?', 'Adim adim:\n1. Geleneksel dokuma\n2. Yun pamuk keten\n3. Sirt sirt gecme\n4. Konya Kars Milas\n5. UNESCO Mirasi')]
for q,a in kultur * 50: add(q,a)
print('  Kultur: ' + str(len(kultur)*50))

random.shuffle(all_data)
val_n = int(len(all_data) * 0.10)
val_data = all_data[:val_n]
train_data = all_data[val_n:]
print('\nVERI: T=' + str(len(train_data)) + ' V=' + str(len(val_data)) + ' Toplam=' + str(len(all_data)))

In [ ]:
# CELL 6: MODEL LOAD
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print('Yukleniyor...')
t0 = time.time()
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained('microsoft/Phi-4-mini-instruct', trust_remote_code=True, token=HF_TOKEN or None)
tok.pad_token = tok.eos_token
tok.padding_side = 'right'
model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-4-mini-instruct', quantization_config=bnb_cfg, device_map='auto', trust_remote_code=True, token=HF_TOKEN or None, torch_dtype=torch.float16, attn_implementation='eager', use_cache=False, low_cpu_mem_usage=True)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
lora_cfg = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], bias='none', task_type='CAUSAL_LM', inference_mode=False)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print('Yukleme: ' + str(int(time.time()-t0)) + 's')

In [ ]:
# CELL 7: TRAIN
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import time
print('Egitim...')
t0 = time.time()
train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)
def fmt(ex):
    texts = []
    for i in range(len(ex['q'])):
        texts.append('<|im_start|>system\n' + SYSTEM_PROMPT + '<|im_end|>\n<|im_start|>user\n' + ex['q'][i] + '<|im_end|>\n<|im_start|>assistant\n' + ex['a'][i] + '<|im_end|>')
    return {'text': texts}
train_ds = train_ds.map(fmt, batched=True, batch_size=2000)
val_ds = val_ds.map(fmt, batched=True, batch_size=2000)
args = SFTConfig(output_dir='/kaggle/working/output', num_train_epochs=3, per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.05, weight_decay=0.01, max_grad_norm=1.0, optim='adamw_torch', fp16=True, max_seq_length=1024, gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False}, logging_steps=25, save_steps=500, save_total_limit=2, eval_strategy='steps', eval_steps=250, load_best_model_at_end=False, report_to='none', seed=42)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, tokenizer=tok, dataset_text_field='text')
model.train()
result = trainer.train()
print('TAMAM ' + str(round((time.time()-t0)/60,1)) + 'dk Loss=' + str(round(result.training_loss, 4)))

In [ ]:
# CELL 8: EVALUATION
import torch, math
from collections import Counter
import numpy as np
print('=== DEGERLENDIRME ===')
model.eval()
def gen(q, max_tok=384):
    prompt = '<|im_start|>system\n' + SYSTEM_PROMPT + '<|im_end|>\n<|im_start|>user\n' + q + '<|im_end|>\n<|im_start|>assistant\n'
    inp = tok(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_tok, temperature=0.7, do_sample=True, top_p=0.9, repetition_penalty=1.1)
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
print('\n--- PERPLEXITY ---')
for t in ['Turkiye nin baskenti Ankara dir.', 'Fotosentez de bitkiler isigi enerjeye cevirir.', 'Python da liste olusturmak icin koseeli parantez kullanilir.']:
    try:
        enc = tok(t, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
        with torch.no_grad():
            out = model(enc['input_ids'], labels=enc['input_ids'])
        print('  PPL=' + str(round(math.exp(out.loss.item()),2)) + ' | ' + t[:50])
    except: print('  PPL hatasi')
print('\n--- BLEU ---')
for q, ref in [('Turkiye baskenti?', 'Turkiye baskenti Ankara'), ('Python liste?', 'sorted liste siralar'), ('Ataturk?', 'Ataturk TC kurucusu')]:
    hyp = gen(q, 128)
    rt = ref.lower().split()
    ht = hyp.lower().split()
    if len(ht) == 0:
        b = 0
    else:
        ps = []
        for i in range(1, 3):
            rng = Counter(tuple(rt[j:j+i]) for j in range(len(rt)-i+1))
            hng = Counter(tuple(ht[j:j+i]) for j in range(len(ht)-i+1))
            ov = sum((hng & rng).values())
            tot = max(len(ht)-i+1, 1)
            ps.append(ov/tot if tot > 0 else 0)
        b = float(np.exp(sum(math.log(p) for p in ps)/2)) if all(p > 0 for p in ps) else 0
    print('  BLEU=' + str(round(b,3)) + ' | ' + q + ' -> ' + hyp[:50])
print('\n--- HALLUSYASYON ---')
for q, keys in [('Fransa baskenti?', ['Paris']), ('1+1?', ['2','iki']), ('En buyuk gezegen?', ['Jupiter']), ('1 yil kac gun?', ['365']), ('Su formul?', ['H2O','H2O'])]:
    hyp = gen(q, 128)
    ok = any(k.lower() in hyp.lower() for k in keys)
    print('  ' + ('OK' if ok else 'FAIL') + ' | ' + q + ' -> ' + hyp[:60])
print('\n--- UZUN CEVAP ---')
lr = gen('Yapay zeka nedir? Detayli acikla.', 512)
lt = len(tok.encode(lr))
print('  Token: ' + str(lt) + ' | ' + ('OK' if lt > 200 else 'KISA'))
print('  ' + lr[:200] + '...')
print('\nTESTLER TAMAM')

In [ ]:
# CELL 9: PUSH (adapter only - T4 icin guvenli)
HUB_ID = 'KuroNeko1234t/phi4-mini-tr'
if HF_TOKEN:
    print('Push...')
    model.save_pretrained('/kaggle/working/adapter')
    tok.save_pretrained('/kaggle/working/adapter')
    from huggingface_hub import create_repo
    create_repo(HUB_ID, token=HF_TOKEN, exist_ok=True, repo_type='model', private=False)
    model.push_to_hub(HUB_ID, token=HF_TOKEN)
    tok.push_to_hub(HUB_ID, token=HF_TOKEN)
    print('https://huggingface.co/' + HUB_ID)
else:
    print('HF_TOKEN yok, push atlandi')
    print('Adapter kaydedildi: /kaggle/working/adapter')
print('TAMAM!')